<a href="https://colab.research.google.com/github/ZeinabEltaib534/Simple-RAG-Customer-Support/blob/main/Customer_SupportRAGProject_depi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df=pd.read_csv('/content/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv.xls')
df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


data preprocessing

In [2]:
df.shape

(26872, 5)

In [3]:
df=df[['instruction','response']]
df.dropna(inplace=True)

In [4]:
df.drop_duplicates()
df.head()

,instruction,response
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...


1-*emdedding*

In [5]:
from sentence_transformers import SentenceTransformer
embed_model=SentenceTransformer('all-MiniLM-L6-v2')
instructions=df['instruction'].tolist()
embeddings=embed_model.encode(instructions,show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/840 [00:00<?, ?it/s]

In [6]:
!pip install faiss-cpu
import faiss
import numpy as np


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.3 MB/s eta 0:00:00


In [7]:
dimensions=embeddings.shape[1]
index=faiss.IndexFlatL2(dimensions)
index.add(np.array(embeddings).astype("float32"))

In [8]:
def retrieve (query, k =3 ):
  query_vec = embed_model.encode([query])
  query_vec= np.array(query_vec).astype("float32")
  D, I  = index.search (query_vec, k)
  #I=[[15,3,4]]
  results=df.iloc[I[0]]
  #I=[15,3,4]
  return results

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = gen_model.generate(**inputs, max_new_tokens=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
#RAG
def rag_pipeline(query):
  retrieved = retrieve(query , k=3 )
  context = retrieved['response'].iloc[0]

  prompt = f"""
  Answer The Question using the context below.
  context:
  {context}

  Question:
  {query}

  Answer:
     """
  answer = generate_answer(prompt)

  return answer

In [11]:
rag_pipeline("cancel order")

"I've realized that you're seeking assistance in canceling an order"

In [12]:
!pip install pyngrok pydeck streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 77.5 MB/s eta 0:00:00


In [13]:
from pyngrok import ngrok
ngrok.set_auth_token("35uLRYnpDiWFrJzrgDjqs7hN3pd_5qov5SGCc12dMtsHCJpgK")

Build Streamlit app

In [14]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re


# Load Models
@st.cache_resource
def load_models():
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    llm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    return embed_model, tokenizer, llm_model


# Load Data
@st.cache_data
def load_data():
    df = pd.read_csv('/content/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv.xls')
    df = df[['instruction', 'response']].dropna().drop_duplicates()
    return df


# Build Index
@st.cache_resource
def build_index(df, _model):
    embeddings = _model.encode(df['instruction'].tolist())
    embeddings = np.array(embeddings).astype("float32")

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return index


# Retrieve
def retrieve(query, model, index, df, k=3):
    query_vec = model.encode([query]).astype("float32")
    D, I = index.search(query_vec, k)
    return df.iloc[I[0]]

# RAG Pipeline
def rag_pipeline(query, embed_model, index, df, tokenizer, llm_model):

    retrieved = retrieve(query, embed_model, index, df, k=3)
    contexts = " ".join(retrieved['response'].tolist())

    prompt = f"""
You are a professional customer support assistant.
Provide a clear, detailed, and complete answer based ONLY on the context below. Include steps if available.

Context:
{contexts[:1500]}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        repetition_penalty=1.2
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    if len(answer) < 30 or query.lower() in answer.lower():
        answer = max(retrieved['response'].tolist(), key=len)

    answer = answer.replace("\n", " ")
    answer = re.sub(r"\{\{.*?\}\}", "", answer)

    return answer


# UI
st.set_page_config(page_title="S3 Chatbot", page_icon="🤖")
st.title("🤖 Customer Support Chatbot")

embed_model, tokenizer, llm_model = load_models()
df = load_data()
index = build_index(df, embed_model)

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input("Ask anything ....."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            response = rag_pipeline(prompt, embed_model, index, df, tokenizer, llm_model)
            st.markdown(response)

    st.session_state.messages.append({"role": "assistant", "content": response})

Writing app.py


In [15]:
get_ipython().system_raw('streamlit run app.py &')
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://unexacerbated-entertainedly-messiah.ngrok-free.dev" -> "http://localhost:8501"
